In [2]:
import sys
sys.path.insert(0, '/Users/anastasiiapovolotskaia/Desktop/OTUS/MLOps_code/OTUS-MLOps-2026-03_HW/hw_04/.venv3.11/lib/python3.11/site-packages')

In [3]:
import os
from datetime import datetime

import pandas as pd
from feast import FeatureStore

In [4]:
raw_data_path = os.path.join("feature_store", "feature_repo", "data", "driver_stats.parquet")
feature_store_path = os.path.join("feature_store", "feature_repo")

### Check data

In [5]:
df = pd.read_parquet(raw_data_path)

In [6]:
df.dtypes

event_timestamp    datetime64[ns, UTC]
driver_id                        int64
conv_rate                      float32
acc_rate                       float32
avg_daily_trips                  int32
created                 datetime64[us]
dtype: object

In [7]:
df.tail(15)

,event_timestamp,driver_id,conv_rate,acc_rate,avg_daily_trips,created
1793,2024-10-16 23:00:00+00:00,1001,0.208080,0.080719,991,2024-10-17 11:30:07.072
1794,2024-10-17 00:00:00+00:00,1001,0.165402,0.644451,485,2024-10-17 11:30:07.072
1795,2024-10-17 01:00:00+00:00,1001,0.737053,0.472196,12,2024-10-17 11:30:07.072
1796,2024-10-17 02:00:00+00:00,1001,0.533628,0.218989,914,2024-10-17 11:30:07.072
1797,2024-10-17 03:00:00+00:00,1001,0.470388,0.540197,491,2024-10-17 11:30:07.072
1798,2024-10-17 04:00:00+00:00,1001,0.330869,0.508280,419,2024-10-17 11:30:07.072
1799,2024-10-17 05:00:00+00:00,1001,0.034440,0.440122,752,2024-10-17 11:30:07.072
1800,2024-10-17 06:00:00+00:00,1001,0.256659,0.186195,617,2024-10-17 11:30:07.072
1801,2024-10-17 07:00:00+00:00,1001,0.574629,0.017635,664,2024-10-17 11:30:07.072
1802,2024-10-17 08:00:00+00:00,1001,0.570691,0.392751,123,2024-10-17 11:30:07.072


### Features inference

In [8]:
entity_df = pd.DataFrame.from_dict(
    {
        # entity's join key -> entity values
        "driver_id": [1001, 1002, 1003],
        # "event_timestamp" (reserved key) -> timestamps
        "event_timestamp": [
            datetime(2021, 4, 12, 10, 59, 42),
            datetime(2021, 4, 12, 8, 12, 10),
            datetime(2021, 4, 12, 16, 40, 26),
        ],
        # (optional) label name -> label values. Feast does not process these
        "label_driver_reported_satisfaction": [1, 5, 3],
    }
)

In [9]:
entity_df

,driver_id,event_timestamp,label_driver_reported_satisfaction
0,1001,2021-04-12 10:59:42,1
1,1002,2021-04-12 08:12:10,5
2,1003,2021-04-12 16:40:26,3


In [10]:
store = FeatureStore(repo_path=feature_store_path)

In [11]:
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "DriverPerformanceFV:conv_rate",
        "DriverPerformanceFV:acc_rate",
        "DriverActivityFV:avg_daily_trips",
    ],
).to_df()

print("----- Feature schema -----\n")
print(training_df.info())

----- Feature schema -----

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 6 columns):
 #   Column                              Non-Null Count  Dtype              
---  ------                              --------------  -----              
 0   driver_id                           3 non-null      int64              
 1   event_timestamp                     3 non-null      datetime64[ns, UTC]
 2   label_driver_reported_satisfaction  3 non-null      int64              
 3   conv_rate                           3 non-null      float32            
 4   acc_rate                            3 non-null      float32            
 5   avg_daily_trips                     3 non-null      int32              
dtypes: datetime64[ns, UTC](1), float32(2), int32(1), int64(2)
memory usage: 240.0 bytes
None


In [12]:
training_df.head()

,driver_id,event_timestamp,label_driver_reported_satisfaction,conv_rate,acc_rate,avg_daily_trips
0,1001,2021-04-12 10:59:42+00:00,1,0.709758,0.692957,402
1,1002,2021-04-12 08:12:10+00:00,5,0.718295,0.584081,370
2,1003,2021-04-12 16:40:26+00:00,3,0.697411,0.197680,25


### Features View - on demand

In [13]:
entity_df = pd.DataFrame.from_dict(
    {
        # entity's join key -> entity values
        "driver_id": [1001, 1002, 1003],
        # "event_timestamp" (reserved key) -> timestamps
        "event_timestamp": [
            datetime(2021, 4, 12, 10, 59, 42),
            datetime(2021, 4, 12, 8, 12, 10),
            datetime(2021, 4, 12, 16, 40, 26),
        ],
        # (optional) label name -> label values. Feast does not process these
        "label_driver_reported_satisfaction": [1, 5, 3],
    }
)

In [14]:
entity_df.head(5)

,driver_id,event_timestamp,label_driver_reported_satisfaction
0,1001,2021-04-12 10:59:42,1
1,1002,2021-04-12 08:12:10,5
2,1003,2021-04-12 16:40:26,3


In [15]:
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "DriverPerformanceFV:conv_rate",
        "DriverPerformanceFV:acc_rate",
        "DriverActivityFV:avg_daily_trips",
        "compute_driver_scores:efficiency_score",
    ],
).to_df()

In [16]:
training_df

,driver_id,event_timestamp,label_driver_reported_satisfaction,conv_rate,acc_rate,avg_daily_trips,efficiency_score
0,1001,2021-04-12 10:59:42+00:00,1,0.709758,0.692957,402,0.491832
1,1002,2021-04-12 08:12:10+00:00,5,0.718295,0.584081,370,0.419543
2,1003,2021-04-12 16:40:26+00:00,3,0.697411,0.197680,25,0.137865


In [18]:
# Online feature retrieval
online_features = store.get_online_features(
    features=[
        "DriverPerformanceFV:conv_rate",
        "DriverPerformanceFV:acc_rate",
        "DriverActivityFV:avg_daily_trips",
    ],
    entity_rows=[{"driver_id": 1001}, {"driver_id": 1002}],
).to_dict()

print("Online features for drivers 1001, 1002:")
for key, value in online_features.items():
    print(f"{key}: {value}")

Online features for drivers 1001, 1002:
driver_id: [1001, 1002]
acc_rate: [None, None]
conv_rate: [None, None]
avg_daily_trips: [None, None]


In [20]:
# Using Feature Service for consistent feature sets
training_df_v1 = store.get_historical_features(
    entity_df=entity_df,
    features=store.get_feature_service("driver_features_v1")
).to_df()

print("\nFeatures from driver_efficiency service:")
training_df_v1.head()


Features from driver_efficiency service:


,driver_id,event_timestamp,label_driver_reported_satisfaction,conv_rate,acc_rate,avg_daily_trips,efficiency_score
0,1001,2021-04-12 10:59:42+00:00,1,0.709758,0.692957,402,0.491832
1,1002,2021-04-12 08:12:10+00:00,5,0.718295,0.584081,370,0.419543
2,1003,2021-04-12 16:40:26+00:00,3,0.697411,0.197680,25,0.137865


In [21]:
# Get feature view metadata
feature_view = store.get_feature_view("DriverPerformanceFV")
print("\nFeature view metadata:")
print(f"Name: {feature_view.name}")
print(f"Entities: {feature_view.entities}")
print(f"TTL: {feature_view.ttl}")
print(f"Online: {feature_view.online}")
print(f"Features: {[f.name for f in feature_view.features]}")


Feature view metadata:
Name: DriverPerformanceFV
Entities: ['driver']
TTL: 1 day, 0:00:00
Online: True
Features: ['conv_rate', 'acc_rate']


In [22]:
# Get feature view metadata
feature_view = store.get_feature_view("DriverActivityFV")
print("\nFeature view metadata:")
print(f"Name: {feature_view.name}")
print(f"Entities: {feature_view.entities}")
print(f"TTL: {feature_view.ttl}")
print(f"Online: {feature_view.online}")
print(f"Features: {[f.name for f in feature_view.features]}")


Feature view metadata:
Name: DriverActivityFV
Entities: ['driver']
TTL: 1 day, 0:00:00
Online: True
Features: ['avg_daily_trips']


In [23]:
# Using Feature Service for consistent feature sets
training_df_v4 = store.get_historical_features(
    entity_df=entity_df,
    features=store.get_feature_service("driver_features_v1")
).to_df()

print("\nFeatures from driver_features_v1 service:")
training_df_v4.head()


Features from driver_features_v1 service:


,driver_id,event_timestamp,label_driver_reported_satisfaction,conv_rate,acc_rate,avg_daily_trips,efficiency_score
0,1001,2021-04-12 10:59:42+00:00,1,0.709758,0.692957,402,0.491832
1,1002,2021-04-12 08:12:10+00:00,5,0.718295,0.584081,370,0.419543
2,1003,2021-04-12 16:40:26+00:00,3,0.697411,0.197680,25,0.137865
